# Phishing Website Detection using Machine Learning

## Notebook 09: Deployment

### Objective

In this notebook, we will:

- Save the final trained LightGBM model.
- Reload the saved model.
- Verify that the loaded model produces the same predictions.
- Create reusable helper functions for making predictions.
- Prepare the model for future integration into applications such as a Streamlit web app.

By the end of this notebook, our model will be deployment-ready and can be reused whenever predictions are required.

---


## 1. Import Required Libraries

In this section, we import the libraries required for loading the trained model,
saving deployment artifacts, and making predictions on new data.

We use **Joblib** because it is efficient for storing and loading machine learning
models built with scikit-learn and LightGBM.

In [1]:
from sklearn.model_selection import train_test_split

# Standard library
from pathlib import Path

# Model persistence
import joblib

# Data manipulation
import pandas as pd

# Machine learning
from sklearn.metrics import accuracy_score

## 2. Load the Engineered Dataset

Load the engineered phishing website dataset created in the previous notebook (notebook 4).

In [2]:
df = pd.read_csv("../dataset/engineered_phishing_dataset.csv")

## 3. Split Dataset

In [3]:
X = df.drop("phishing", axis=1)
y = df["phishing"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

## 2. Load the Saved LightGBM Model

The final LightGBM model was trained and saved in the previous notebook.


In [4]:
# Load the trained LightGBM model
model = joblib.load("../models/tuned/LightGBM.pkl")

print("Model loaded successfully!")
print(f"Model Type: {type(model)}")

Model loaded successfully!
Model Type: <class 'lightgbm.sklearn.LGBMClassifier'>


## 3. Verify the Loaded Model

After loading the saved model, it is important to verify that it has been restored correctly.

One simple way to do this is by checking some of the model's basic properties, such as its type and important parameters. 

In [5]:
# Display basic information about the loaded model

print("Model Information")

print(f"Model Type      : {type(model).__name__}")
print(f"Model Class     : {model.__class__.__name__}")

# Display important model parameters
print("\nModel Parameters:")

print(f"Learning Rate   : {model.learning_rate}")
print(f"Number of Trees : {model.n_estimators}")
print(f"Maximum Depth   : {model.max_depth}")
print(f"Random State    : {model.random_state}")

Model Information
Model Type      : LGBMClassifier
Model Class     : LGBMClassifier

Model Parameters:
Learning Rate   : 0.1
Number of Trees : 100
Maximum Depth   : 6
Random State    : 42


## 5. Create Helper Prediction Functions

The following helper functions will be implemented:

- **predict_website()** – Predicts whether a website is legitimate or phishing.
- **predict_probability()** – Returns the probability for each prediction.

These helper functions improve code readability and can be reused later in the Streamlit web application.

### 5.1 Predict Website Class

This helper function predicts the class label for one or more website samples using the trained LightGBM model.

The function returns:

- **0** → Legitimate Website
- **1** → Phishing Website

In [6]:
def predict_website(model, input_data):
    """
    Predict whether a website is legitimate or phishing.

    Parameters
    ----------
    model : LightGBM Classifier
        Trained machine learning model.

    input_data : pandas.DataFrame
        Input features for prediction.

    Returns
    -------
    numpy.ndarray
        Predicted class labels.
    """
    return model.predict(input_data)

### 5.2 Predict Class Probabilities

Besides predicting the class label, it is often useful to know how confident the model is in its prediction.

This helper function returns the prediction probabilities for both classes.

In [7]:
def predict_probability(model, input_data):
    """
    Predict class probabilities.

    Parameters
    ----------
    model : LightGBM Classifier
        Trained machine learning model.

    input_data : pandas.DataFrame
        Input features for prediction.

    Returns
    -------
    numpy.ndarray
        Prediction probabilities.
    """
    return model.predict_proba(input_data)

## 6. Test the Prediction Pipeline

Before integrating the model into an application, it is important to verify that the prediction pipeline works correctly.

In this section, we will use a few samples from the test dataset to:

- Predict whether a website is legitimate or phishing.
- Calculate the prediction probabilities.
- Compare the predicted labels with the actual labels.

Testing the prediction pipeline helps ensure that the saved model is functioning correctly and produces reliable predictions.

### 6.1 Select Sample Test Data

To demonstrate the prediction process, a small subset of the test dataset is selected.

Using a few samples keeps the output simple while allowing us to verify that the deployment pipeline is working correctly.

In [8]:
# Select the first five samples from the test dataset
sample_data = X_test.iloc[:5]
actual_labels = y_test.iloc[:5]

sample_data

,num_dots_url,num_hyph_url,num_underline_url,num_slash_url,num_questionmark_url,num_equal_url,at_sign_url,num_exclamation_url,num_space_url,tilde_url,...,num_questionmark_param,at_sign_param,num_exclamation_param,num_space_param,tilde_param,num_comma_param,num_plus_param,num_asterisk_param,num_percent_param,tld_in_param
3201,2,1,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1359,1,1,0,5,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1251,4,0,0,3,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1216,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1094,2,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### 6.2 Predict Website Class

The selected samples are passed to the `predict_website()` helper function to obtain the predicted class labels.

The predicted labels are then compared with the actual labels from the test dataset.

In [9]:
# Generate predictions
predictions = predict_website(model, sample_data)

# Display prediction results
prediction_df = pd.DataFrame({
    "Actual Label": actual_labels.values,
    "Predicted Label": predictions
})

prediction_df

,Actual Label,Predicted Label
0,1,1
1,1,1
2,0,0
3,0,1
4,1,1


### Interpretation

- The loaded LightGBM model successfully generated predictions for the selected test samples.
- Most predictions match the actual labels, indicating that the model is working correctly.
- The model is ready to make predictions on new, unseen data.

### 6.3 Predict Class Probabilities

In addition to predicting the class label, the model can estimate the probability of each class.

Prediction probabilities provide insight into the model's confidence. A higher probability indicates greater confidence in the corresponding prediction.

In [10]:
# Predict class probabilities
probabilities = predict_probability(model, sample_data)

# Display probabilities
probability_df = pd.DataFrame(
    probabilities,
    columns=["Legitimate Probability", "Phishing Probability"]
)

probability_df

,Legitimate Probability,Phishing Probability
0,0.409338,0.590662
1,0.004233,0.995767
2,0.737046,0.262954
3,0.400525,0.599475
4,0.031864,0.968136


### Interpretation

- The prediction probabilities indicate the model's confidence for each prediction.
- Higher probability values represent greater confidence in the predicted class.
- These probabilities provide additional insight beyond the predicted class label.

### 6.4 Display Complete Prediction Results

To better understand the prediction process, the actual labels, predicted labels, and prediction probabilities are combined into a single table.

This provides a complete overview of the model's predictions for the selected test samples.

In [11]:
# Combine all prediction results
results_df = sample_data.copy()

results_df["Actual Label"] = actual_labels.values
results_df["Predicted Label"] = predictions
results_df["Legitimate Probability"] = probabilities[:, 0]
results_df["Phishing Probability"] = probabilities[:, 1]

results_df

,num_dots_url,num_hyph_url,num_underline_url,num_slash_url,num_questionmark_url,num_equal_url,at_sign_url,num_exclamation_url,num_space_url,tilde_url,...,tilde_param,num_comma_param,num_plus_param,num_asterisk_param,num_percent_param,tld_in_param,Actual Label,Predicted Label,Legitimate Probability,Phishing Probability
3201,2,1,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,1,1,0.409338,0.590662
1359,1,1,0,5,0,0,0,0,0,0,...,0,0,0,0,0,0,1,1,0.004233,0.995767
1251,4,0,0,3,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0.737046,0.262954
1216,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0.400525,0.599475
1094,2,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,1,1,0.031864,0.968136


### Interpretation

- The combined results provide a complete overview of the prediction process.
- The model correctly classified most of the selected samples.
- This confirms that the deployment pipeline is functioning correctly and is ready for future applications.

---

## 7. Organize Deployment Files

In addition to the trained model, deployment often requires storing other supporting files that help maintain a consistent prediction pipeline.

One important file is the list of feature names used during model training. Saving the feature names ensures that new input data follows the same structure as the training data, reducing the risk of prediction errors.

In this section, we will save the feature names for future use during deployment.

### 7.1 Save Feature Names

The trained LightGBM model expects the input features to have the same order and names as the training dataset.

To maintain consistency, the feature names are saved as a separate deployment artifact.

In [12]:
# Define the feature file path
FEATURE_PATH = Path("../models/feature_columns.pkl")

# Save feature names
joblib.dump(list(X_train.columns), FEATURE_PATH)

print(f"Feature names saved successfully at:\n{FEATURE_PATH}")

Feature names saved successfully at:
..\models\feature_columns.pkl


### 7.2 Verify Saved Feature Names

To verify that the feature names have been stored correctly, they are loaded back from disk and displayed.

In [13]:
# Load saved feature names
feature_columns = joblib.load(FEATURE_PATH)

print(f"Total Features: {len(feature_columns)}")
print("\nFirst 10 Features:")
print(feature_columns[:10])

Total Features: 56

First 10 Features:
['num_dots_url', 'num_hyph_url', 'num_underline_url', 'num_slash_url', 'num_questionmark_url', 'num_equal_url', 'at_sign_url', 'num_exclamation_url', 'num_space_url', 'tilde_url']


## 8. Deployment Pipeline Overview

The deployment workflow followed in this notebook can be summarized as the following pipeline:

```
Training
     │
     ▼
Save Trained LightGBM Model
     │
     ▼
Load Saved Model
     │
     ▼
Create Helper Prediction Functions
     │
     ▼
Predict Website Class
     │
     ▼
Generate Prediction Probabilities
     │
     ▼
Ready for Deployment
```

This pipeline ensures that the trained model can be reused efficiently without retraining. The saved model and feature information are now ready to be integrated into applications such as a Streamlit web app.

## 9. Key Findings

The following are the key outcomes of this notebook:

- Successfully loaded the saved LightGBM model.
- Verified that the loaded model produces valid predictions.
- Created reusable helper functions for prediction.
- Tested the prediction pipeline using sample test data.
- Saved the feature names required for future predictions.
- Prepared the model for integration into deployment applications.

## 10. Conclusion

In this notebook, the trained LightGBM model was successfully prepared for deployment. The saved model was loaded, verified, and tested using unseen samples to ensure that it generates reliable predictions.

Reusable helper functions were created to simplify the prediction workflow, and the feature names were saved to maintain consistency during future inference.

With the deployment artifacts ready, the project is now prepared for the next stage, where the model will be integrated into a **Streamlit web application** for interactive phishing website detection.